# Challenge 4: Programming an Agent Workflow

# **1 | Install Dependencies**

In [ ]:
!pip install "google-adk[extensions]" google-cloud-aiplatform vertexai requests --quiet

# **2 | Imports and Configuration**

In [ ]:
import os
import asyncio
import getpass
from typing import Optional

from google.adk.agents import Agent, SequentialAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.tools.google_search_tool import google_search
from google.genai import types

PROJECT_ID = "qwiklabs-gcp-01-ab542815eb6c"
LOCATION = "us-central1"

# Use Vertex AI credentials instead of a standalone Gemini API key
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

MODEL_GEMINI = "gemini-2.5-flash"

print("Configuration complete.")

# **3 | Initialize Vertex AI**

In [ ]:
import google.auth
import vertexai

credentials, project = google.auth.default()
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Authenticated as project: {project or PROJECT_ID}")

# **4 | Define Callbacks**

In [ ]:
def log_before(
    callback_context: CallbackContext,
    llm_request: LlmRequest,
) -> Optional[LlmResponse]:
    """Log a preview of the input before it is sent to the model."""
    user_text = ""
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.parts:
            user_text = last.parts[0].text or ""
    print(f"[{callback_context.agent_name} \u2192 BEFORE MODEL] Input: {user_text[:120]!r}")
    return None


def log_after(
    callback_context: CallbackContext,
    llm_response: LlmResponse,
) -> Optional[LlmResponse]:
    """Log tool calls or a response preview after the model responds."""
    tool_calls = []
    response_text = ""
    if llm_response.content and llm_response.content.parts:
        for part in llm_response.content.parts:
            if hasattr(part, "text") and part.text:
                response_text = part.text
            if hasattr(part, "function_call") and part.function_call:
                tool_calls.append(part.function_call.name)
    if tool_calls:
        print(f"[{callback_context.agent_name} \u2192 AFTER MODEL] Tool call(s): {tool_calls}")
        return None
    if not response_text:
        return None
    preview = response_text[:200] + "..." if len(response_text) > 200 else response_text
    print(f"[{callback_context.agent_name} \u2192 AFTER MODEL] Response: {preview!r}")
    return None


print("Callbacks defined.")

# **5 | Agent Instructions**

In [ ]:
SEARCH_AGENT_INSTRUCTIONS = """
You are a research agent. Your job is to find accurate, up-to-date information
to answer the user's question.

Today's date is September 25, 2026.

Steps:
1. Use the google_search tool to find relevant information
2. Summarize the key facts clearly and concisely
3. When discussing events or topics, treat anything before September 25, 2026 as
   already having occurred — do not describe past events as future or upcoming
4. Always cite what you found — do not make up information

Return a clear, factual answer that other agents can build on.
"""

CRITIQUE_AGENT_INSTRUCTIONS = """
You are a critical review agent. You will receive a draft answer to a question.

Today's date is September 25, 2026. Events before this date have already occurred.
Do not flag past events as "future" or "upcoming".

Your job:
1. Identify any factual gaps, missing context, or unclear explanations
2. Note anything that could be more accurate, complete, or better structured
3. Suggest specific improvements — be constructive and precise

Do NOT rewrite the answer. Only provide a critique with actionable suggestions.
Return your critique as a numbered list.
"""

REFINE_AGENT_INSTRUCTIONS = """
You are a refinement agent. You will receive:
- An original answer to a question
- A critique with suggestions for improvement

Your job:
1. Rewrite the answer incorporating all valid suggestions from the critique
2. Make it clear, accurate, well-structured, and complete
3. Do not introduce new information beyond what the search found

Return only the final polished answer — no preamble, no meta-commentary.
"""

ROOT_AGENT_INSTRUCTIONS = """
You are Erwin, a helpful assistant that answers questions thoroughly and accurately.

For every question you receive:
1. Pass it to the Erwin_AnswerTeam to research, critique, and refine the response
2. Return the final refined answer to the user

Always delegate to Erwin_AnswerTeam — do not answer directly yourself.
"""

print("Instructions defined.")

# **6 | Build the Search Agent**

In [ ]:
search_agent = Agent(
    name="Erwin_Search",
    model=MODEL_GEMINI,
    description="Searches the web to find accurate answers to questions.",
    instruction=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
    before_model_callback=log_before,
    after_model_callback=log_after,
)

print("Search agent created.")

# **7 | Build the Critique Agent**

In [ ]:
critique_agent = Agent(
    name="Erwin_Critique",
    model=MODEL_GEMINI,
    description="Reviews a draft answer and provides suggestions for improvement.",
    instruction=CRITIQUE_AGENT_INSTRUCTIONS,
    before_model_callback=log_before,
    after_model_callback=log_after,
)

print("Critique agent created.")

# **8 | Build the Refine Agent**

In [ ]:
refine_agent = Agent(
    name="Erwin_Refine",
    model=MODEL_GEMINI,
    description="Rewrites and polishes an answer based on critique feedback.",
    instruction=REFINE_AGENT_INSTRUCTIONS,
    before_model_callback=log_before,
    after_model_callback=log_after,
)

print("Refine agent created.")

# **9 | Build the Answer Team (SequentialAgent)**

In [ ]:
answer_team = SequentialAgent(
    name="Erwin_AnswerTeam",
    description="A sequential workflow that searches, critiques, and refines answers.",
    sub_agents=[search_agent, critique_agent, refine_agent],
)

print("Answer team (SequentialAgent) created.")

# **10 | Build the Root Agent**

In [ ]:
root_agent = Agent(
    name="Erwin_Root",
    model=MODEL_GEMINI,
    description="Receives user questions and delegates to the Erwin_AnswerTeam workflow.",
    instruction=ROOT_AGENT_INSTRUCTIONS,
    sub_agents=[answer_team],
    before_model_callback=log_before,
    after_model_callback=log_after,
)

print("Root agent created.")

# **11 | Helper: Run Agent**

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from IPython.display import Markdown, display

async def run_agent(agent, query: str, user_id: str = "test-user") -> str:
    """Run a query through the ADK Runner and return the final text response.
    Automatically retries on 429 rate limit errors with exponential backoff.
    """
    RETRY_DELAYS = [15, 30, 60]

    for attempt, delay in enumerate([0] + RETRY_DELAYS):
        if delay > 0:
            print(f"  [429 Rate limit hit — retrying in {delay}s (attempt {attempt}/{len(RETRY_DELAYS)})...]")
            await asyncio.sleep(delay)

        try:
            session_service = InMemorySessionService()
            runner = Runner(
                agent=agent,
                app_name=agent.name,
                session_service=session_service,
            )
            session = await session_service.create_session(
                app_name=agent.name,
                user_id=user_id,
            )
            content = types.Content(
                role="user",
                parts=[types.Part(text=query)],
            )
            response_text = ""
            async for event in runner.run_async(
                user_id=user_id,
                session_id=session.id,
                new_message=content,
            ):
                if event.is_final_response() and event.content and event.content.parts:
                    response_text = event.content.parts[0].text
            return response_text or "No response received."

        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                if attempt < len(RETRY_DELAYS):
                    continue
                else:
                    return "Error: Rate limit exceeded after all retries. Please wait a minute and try again."
            raise


print("run_agent helper defined (with 429 retry logic).")

# **12 | Test: Workflow Execution (Search \u2192 Critique \u2192 Refine)**

In [ ]:
test_queries = [
    "What is the Google Agent Development Kit and what are its main features?",
    "Who won the most recent Super Bowl and what was the final score?",
]

print("=" * 60)
print("TEST: ANSWER WORKFLOW \u2014 SEARCH \u2192 CRITIQUE \u2192 REFINE")
print("=" * 60)

for query in test_queries:
    print(f"\nQuery: {query}")
    print("-" * 40)
    response = await run_agent(root_agent, query)
    display(Markdown(response))
    print()

# **13 | Interactive Chat**

In [ ]:
import nest_asyncio
import asyncio

nest_asyncio.apply()

print("\u250c" + "\u2500" * 47 + "\u2510")
print("\u2502       Erwin \u2014 Answer Workflow Assistant       \u2502")
print("\u2514" + "\u2500" * 47 + "\u2518")
print("Ask me anything \u2014 I'll search, critique, and refine before answering.")
print("Type 'quit' to end.\n")

loop = asyncio.get_event_loop()

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nErwin: Goodbye!")
        break

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit", "q", "bye"):
        print("Erwin: Goodbye!")
        break

    print("  [Searching, critiquing, refining...]\n")
    response = loop.run_until_complete(run_agent(root_agent, user_input))
    display(Markdown(f"**Erwin:** {response}"))
    print()